In [1]:
import pandas as pd
from datasets import Dataset
df = pd.read_csv('/home/wagyu0923/project/Document_Analyzer/evaluation_data.csv')


/home/wagyu0923/miniconda3/envs/exaone/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
import os
sys.path.append('/home/wagyu0923/project/Document_Analyzer')
from pipeline.document_loader import DocumentLoader
from pipeline.chunker import Chunker
from pipeline.embedder import Embedder
from pipeline.vector_retriever import VectorRetriever
from pipeline.generator import Generator
import config
from tkinter import filedialog

def setup_pipeline():
    chunker = Chunker(
        chunk_size = config.CHUNK_SIZE,
        overlap_size = config.OVERLAP_SIZE
    )
    print('Chunking Complete')
    embedder = Embedder(
        model_name = config.EMBEDDING_MODEL
    )
    print('Embedding Complete')
    retriever =VectorRetriever(
        db_path = config.DB_PATH,
        model_name = config.EMBEDDING_MODEL,
        collection_name = config.COLLECTION_NAME
    )
    print('Retrieving Coplete')
    generator = Generator(
        model_name = config.LLM_NAME,
        options = config.DEFAULT_OLLAMA_OPTIONS
    )
    return chunker, embedder, retriever, generator

def run_indexing(file_path, chunker, embedder, retriever):
    loader = DocumentLoader(file_path = file_path)
    document = loader.load()
    chunks = chunker.chunking(document)
    embedded_chunks = embedder.embed_documents(chunks)
    file_name = os.path.basename(file_path)
    retriever.add_documents(embedded_chunks, file_name)

chunker, embedder, retriever, generator = setup_pipeline()

file_path = '/home/wagyu0923/project/Document_Analyzer/pdf_files/[세토피아][정정]반기보고서(2025.09.09).pdf'
run_indexing(file_path, chunker, embedder, retriever)

Chunking Complete
Embedding Complete
Retrieving Coplete


In [ ]:
import json
dataset = df.copy()
for index, query in enumerate(df['user_input']):
    retrieved_data = retriever.retrieve(query, n_results = 5)
    outputs = generator.generate(retrieved_data, query)
    try:
        outputs = json.loads(outputs)
    except json.JSONDecodeError:
        print(f'JSON Decode Error at index {index+1}. Skipping.') 
        print(outputs)
        continue 
    
    if 'answer' not in outputs.keys():
        outputs['answer'] = ''
    dataset.loc[index, 'retrieved_contexts'] = retrieved_data
    dataset.loc[index, 'response'] = outputs['answer']
    print(f'progress : {index+1}/{len(df)}')
    print(outputs)




In [15]:
import ast  
import json 
import pandas as pd 

dataset["retrieved_contexts"] = dataset["retrieved_contexts"].apply(
    lambda x: []
    if x is None or (isinstance(x, float) and pd.isna(x)) or x == ""
    else (ast.literal_eval(x.strip()) if isinstance(x, str) and x.strip().startswith("[") and x.strip().endswith("]")
          else (x if isinstance(x, list) else [x]))
)

if "reference" in dataset.columns:
    dataset["reference"] = dataset["reference"].apply(
        lambda x: "" if x is None or (isinstance(x, float) and pd.isna(x))
        else (x if isinstance(x, str) else "\n\n".join(map(str, x)))
    )
    
dataset.fillna('',inplace=True)

In [16]:
import re

def strip_source_block(text: str) -> str:
    if not isinstance(text, str):
        return text

    blocks = text.split("source :")
    contents = []

    for block in blocks:
        block = block.strip()
        if not block:
            continue

    
        if "content :" in block:
            content_part = block.split("content :", 1)[1].strip()
            contents.append(content_part)
        else:
        
            contents.append(block)


    return "\n\n".join(contents)


def clean_retrieved_contexts(x):
   
    if isinstance(x, list):
        return [strip_source_block(t) for t in x]
    
    elif isinstance(x, str):
        return [strip_source_block(x)]

    else:
        return []



dataset["retrieved_contexts"] = dataset["retrieved_contexts"].apply(clean_retrieved_contexts)

In [21]:
dataset.to_csv('response_data.csv')

In [23]:
import os
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

from ragas import EvaluationDataset, evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
)
from ragas.run_config import RunConfig


evaluation_dataset = EvaluationDataset.from_pandas(dataset)


from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper

base_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key = OPENAI_API_KEY        
)

evaluator_llm = LangchainLLMWrapper(base_llm)


metrics = [
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
]

result = evaluate(
    dataset=evaluation_dataset,
    metrics=metrics,
    llm=evaluator_llm,         
)

print(result)


/tmp/ipykernel_29737/842449243.py:28: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(base_llm)
Evaluating:  22%|██▏       | 26/120 [00:58<02:37,  1.67s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  33%|███▎      | 40/120 [01:19<01:46,  1.33s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  50%|█████     | 60/120 [01:54<01:27,  1.45s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generatio

{'context_precision': 0.8333, 'context_recall': 0.7583, 'faithfulness': 0.7816, 'answer_relevancy': 0.4467}


In [25]:
import pandas as pd


result_df = result.to_pandas()
result_df.to_csv('result.csv')
result_df

,user_input,retrieved_contexts,response,reference,context_precision,context_recall,faithfulness,answer_relevancy
0,(주)세토피아의 현재 대표이사는 누구인가?,[【 대표이사 등의 확인 】 전자공시시스템 dart.fss.or.kr Page 3반...,서상철,(주)세토피아의 현재 대표이사는 서상철입니다.,1.0,1.00,1.000000,0.796597
1,(주)세토피아의 2022년 8월 1일 이전 상호명은 무엇이었나?,"[6,899 1,521,181,835 33,078,491,588 부채와 자본총계 2...",(주)마이더스AI,(주)마이더스AI였습니다.,1.0,1.00,1.000000,0.769330
2,(주)세토피아가 (주)제이슨앤컴퍼니를 흡수합병한 합병기일은 언제인가?,[갈음할 예정입니다. (3) 합병계약 체결일로부터 합병등기일까지 사이에 관계기관과의...,2023.01.02,2023년 1월 2일입니다.,1.0,1.00,1.000000,0.746207
3,세토피아 철강사업의 주력 제품과 국내 시장 점유율은 어떻게 되는가?,[http://www.se-topia.com 라. 중소기업 등 해당 여부 마. 주요...,제공된 컨텍스트만으로는 질문에 답변할 수 없습니다.,"주력 제품은 STS 201이며, 국내 시장점유율 약 30~40%로 1위를 차지하고 ...",0.0,0.00,0.000000,0.000000
4,세토피아가 진출했던 유통사업은 어떤 브랜드 제품의 국내 총판 독점 계약이었나?,"[6,899 1,521,181,835 33,078,491,588 부채와 자본총계 2...",ELFBAR(엘프바) 전자담배브랜드,글로벌 1위 전자담배브랜드 'ELFBAR(엘프바)' 제품의 국내 총판 독점 계약이었...,1.0,1.00,1.000000,0.674993
5,"제11기(당반기) 감사보고서의 감사의견은 무엇이며, 감사인은 누구인가?",[감사의견 관련 커뮤니케이션 사업연도 감사인 검토의견 지적사항 제11기(당기) (2...,"제11기(당반기) 감사보고서의 감사의견은 의견거절이며, 감사인은 삼일회계법인입니다.","감사인은 삼일회계법인이며, 감사의견은 '의견거절'입니다.",1.0,1.00,1.000000,0.791447
6,2025년 1월 20일 (주)에스에이코퍼레이션으로부터의 유형자산(토지 및 건물) 양...,[통해 매도인과의 합의하여 2025년 01월 20일에 본 계약을 해지 하는 것으로 ...,2025년 1월 20일 (주)에스에이코퍼레이션으로부터의 유형자산(토지 및 건물) 양...,2025년 1월 20일부로 해당 계약을 해지(철회)하였습니다.,1.0,1.00,1.000000,0.851239
7,2025년 2월 12일에 완료된 무상감자의 비율은 어떻게 되는가?,"[13,332,272 88.11% 합 계 15,131,870 100.00% 전자공시...",제공된 컨텍스트만으로는 질문에 답변할 수 없습니다.,5:1 무상감자를 완료하였습니다.,0.0,0.00,1.000000,0.000000
8,"2025년 6월 30일 기준, 세토피아의 최대주주는 누구이며 지분율은 몇 %인가?","[6,899 1,521,181,835 33,078,491,588 부채와 자본총계 2...",제공된 컨텍스트만으로는 질문에 답변할 수 없습니다.,"최대주주는 (주)에스에이코퍼레이션이며, 지분율은 6.89%입니다.",1.0,1.00,0.000000,0.000000
9,"2025년 3월 12일 증권선물위원회가 세토피아에 부과한 과징금 금액은 얼마이며, ...",[로부터 받은 제재 일자 제재기관 대상자 처벌 또는 조치 내용 금전적 제재금액 사유...,"2025년 3월 12일 증권선물위원회가 세토피아에 부과한 과징금은 2.7억원이며, ...","과징금 2.7억원이 부과되었으며, 주된 사유는 금융자산·부채 과대계상(2019년 8...",1.0,1.00,1.000000,0.824996
